# 03 – Model Experiments
In this notebook, we run experiments across 4 compact models to compare their performance on the fraud detection dataset using a validation split.

## Step 1: Imports

In [9]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from preprocessing import clean_data

## Step 2: Load Data and Create Train/Test Split

In [10]:
df = pd.read_csv('../Data/raw/credit_card_fraud_messy.csv')
df_clean = clean_data(df, drop_duplicates=True)

target = 'is_fraud'
drop_cols = ['transaction_id', 'customer_id', 'transaction_datetime', target]
feature_cols = [c for c in df_clean.columns if c not in drop_cols]

X = df_clean[feature_cols]
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (12000, 22)
Testing shape: (3000, 22)


## Step 3: Define Preprocessing Pipelines

In [11]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

C:\Users\Gehan\AppData\Local\Temp\ipykernel_29248\1101041292.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


## Step 4: Model 1 – Logistic Regression

In [12]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_preds)
print(f"Logistic Regression ROC-AUC: {lr_auc:.4f}")

Logistic Regression ROC-AUC: 0.9370


## Step 5: Model 2 – Decision Tree

In [13]:
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

dt_pipeline.fit(X_train, y_train)
dt_preds = dt_pipeline.predict_proba(X_test)[:, 1]
dt_auc = roc_auc_score(y_test, dt_preds)
print(f"Decision Tree ROC-AUC: {dt_auc:.4f}")

Decision Tree ROC-AUC: 0.8569


## Step 6: Model 3 – Random Forest

In [14]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_preds)
print(f"Random Forest ROC-AUC: {rf_auc:.4f}")

Random Forest ROC-AUC: 0.9354


## Step 7: Model 4 – Gradient Boosting

In [15]:
gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

gb_pipeline.fit(X_train, y_train)
gb_preds = gb_pipeline.predict_proba(X_test)[:, 1]
gb_auc = roc_auc_score(y_test, gb_preds)
print(f"Gradient Boosting ROC-AUC: {gb_auc:.4f}")

Gradient Boosting ROC-AUC: 0.9427


## Step 8: Compare Results & Select Best Model

In [21]:
import pandas as pd

results = {
    'Logistic Regression': lr_auc,
    'Decision Tree': dt_auc,
    'Random Forest': rf_auc,
    'Gradient Boosting': gb_auc
}

print("=== EXPERIMENT SUMMARY ===")

df_results = (
    pd.Series(results, name="ROC-AUC")
    .rename_axis("Model")
    .map("{:.4f}".format)
    .to_frame()
)

display(df_results)

best_model = max(results, key=results.get)
print(f"\nBest Model to Use: {best_model} ({results[best_model]:.4f})")


=== EXPERIMENT SUMMARY ===


,ROC-AUC
Model,
Logistic Regression,0.9370
Decision Tree,0.8569
Random Forest,0.9354
Gradient Boosting,0.9427



Best Model to Use: Gradient Boosting (0.9427)
